# Cleanup: Config and Data Setup

This notebook cleans up resources created by `setup/01_config_and_data_setup.ipynb`.

**Resources that will be deleted:**

**1. Dataplex Aspects:**
   - Remove `census-survey-metadata` and `data-governance-public` aspects from all BigQuery table entries (~278 tables)

**2. Dataplex Aspect Types:**
   - `census-survey-metadata`
   - `data-governance-public`

**3. BigQuery Dataset:**
   - `census_bureau_acs` (and all ~278 tables)

**⚠️  WARNING:**
- This operation cannot be undone
- All data will be permanently deleted
- All custom metadata (aspects) will be removed

**Required permissions:**
- `roles/bigquery.admin` (to delete BigQuery dataset)
- `roles/dataplex.admin` (to delete aspect types)
- `roles/dataplex.catalogAdmin` (to remove aspects from entries)

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
CATALOG_LOCATION = "us"  # BigQuery US multi-region datasets are cataloged in 'us'

# BigQuery dataset
DATASET_ID = "census_bureau_acs"
DATASET_LOCATION = "US"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Catalog Location: {CATALOG_LOCATION}")
print(f"  Dataset to delete: {DATASET_ID}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = ["google-cloud-bigquery", "google-cloud-dataplex"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

In [ ]:
# Initialize clients
from google.cloud import bigquery
from google.cloud import dataplex_v1
import requests
from google.auth.transport.requests import Request

bq_client = bigquery.Client(project=PROJECT_ID)
catalog_client = dataplex_v1.CatalogServiceClient()

# Get credentials for REST API calls
credentials.refresh(Request())

print("✅ Clients initialized:")
print("  - BigQuery client")
print("  - Dataplex Catalog client")
print("  - Authentication credentials")

## Section 2: Remove Aspects from BigQuery Table Entries

Before deleting the aspect types, we need to remove all aspect instances that were applied to BigQuery table entries.

This will:
- List all tables in the census_bureau_acs dataset
- Remove both aspects (census-survey-metadata and data-governance-public) from each table entry

In [ ]:
# List all tables in the dataset
print("📋 Listing tables in dataset...")
print()

dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"

try:
    tables = list(bq_client.list_tables(dataset_ref))
    total_tables = len(tables)
    
    print(f"✅ Found {total_tables} tables in {DATASET_ID}")
    print(f"   Will remove aspects from all {total_tables} table entries")
    print()
    
except Exception as e:
    error_msg = str(e)
    if "404" in error_msg or "not found" in error_msg.lower():
        print(f"⚠️  Dataset {DATASET_ID} not found - skipping aspect removal")
        total_tables = 0
    else:
        print(f"❌ Error listing tables: {error_msg}")
        raise

In [ ]:
# Remove aspects from all table entries using REST API PATCH
print("🗑️  Removing aspects from BigQuery table entries...")
print()

if total_tables == 0:
    print("⚠️  No tables found - skipping aspect removal")
else:
    # Aspect IDs to remove
    aspect_ids = ["census-survey-metadata", "data-governance-public"]
    
    catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
    BQ_ENTRY_GROUP = "@bigquery"
    
    success_count = 0
    not_found_count = 0
    error_count = 0
    errors = []
    
    print(f"⏳ Processing {total_tables} tables...")
    
    headers = {
        "Authorization": f"Bearer {credentials.token}",
        "Content-Type": "application/json"
    }
    
    for i, table in enumerate(tables):
        table_id = table.table_id
        
        # Construct the Dataplex entry name
        entry_id = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
        entry_name = f"{catalog_parent}/entryGroups/{BQ_ENTRY_GROUP}/entries/{entry_id}"
        
        # Get the entry to check which aspects exist
        entry_url = f"https://dataplex.googleapis.com/v1/{entry_name}"
        
        try:
            # Get entry to see current aspects
            get_response = requests.get(entry_url, headers=headers)
            
            if get_response.status_code == 404:
                not_found_count += 1
                continue
            
            if get_response.status_code != 200:
                error_count += 1
                errors.append((table_id, "get", f"Status {get_response.status_code}"))
                continue
            
            entry_data = get_response.json()
            aspects = entry_data.get('aspects', {})
            
            # Check if any aspects need to be removed
            aspects_to_remove = []
            for aspect_id in aspect_ids:
                aspect_key = f"{PROJECT_ID}.{CATALOG_LOCATION}.{aspect_id}"
                if aspect_key in aspects:
                    aspects_to_remove.append(aspect_key)
            
            if not aspects_to_remove:
                not_found_count += len(aspect_ids)
                continue
            
            # Remove the aspects by patching with empty aspects
            new_aspects = {k: v for k, v in aspects.items() if k not in aspects_to_remove}
            
            patch_data = {
                "aspects": new_aspects
            }
            
            patch_url = f"{entry_url}?updateMask=aspects"
            patch_response = requests.patch(patch_url, headers=headers, json=patch_data)
            
            if patch_response.status_code == 200:
                success_count += len(aspects_to_remove)
            else:
                error_count += 1
                errors.append((table_id, "patch", f"Status {patch_response.status_code}"))
                
        except Exception as e:
            error_count += 1
            errors.append((table_id, "exception", str(e)[:60]))
        
        # Show progress every 25 tables
        if (i + 1) % 25 == 0:
            print(f"   Processed {i + 1}/{total_tables} tables... ({success_count} aspects removed)")
    
    print()
    print("=" * 60)
    print("✅ Aspect removal completed!")
    print("=" * 60)
    print(f"   Tables processed: {total_tables}")
    print(f"   Aspects removed: {success_count}")
    print(f"   Not found (already removed): {not_found_count}")
    print(f"   Errors: {error_count}")
    
    if error_count > 0:
        print()
        print(f"⚠️  Errors encountered ({error_count}):")
        for table_id, op, error in errors[:5]:
            print(f"   - {table_id} ({op}): {error}")
        if len(errors) > 5:
            print(f"   ... and {len(errors) - 5} more errors")

## Section 3: Delete Dataplex Aspect Types

Now we'll delete the custom aspect types created for the demo:
- `census-survey-metadata`
- `data-governance-public`

In [ ]:
# Delete aspect types
print("🗑️  Deleting Dataplex Aspect Types...")
print()

aspect_type_ids = ["census-survey-metadata", "data-governance-public"]
catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"

deleted_count = 0
not_found_count = 0
error_count = 0

for aspect_type_id in aspect_type_ids:
    aspect_type_name = f"{catalog_parent}/aspectTypes/{aspect_type_id}"
    
    print(f"🗑️  Deleting: {aspect_type_id}")
    
    try:
        # Delete the aspect type
        delete_operation = catalog_client.delete_aspect_type(name=aspect_type_name)
        
        # Wait for deletion to complete
        delete_operation.result(timeout=300)
        
        print(f"   ✅ Deleted: {aspect_type_id}")
        deleted_count += 1
        
    except Exception as e:
        error_msg = str(e)
        
        if "404" in error_msg or "not found" in error_msg.lower():
            print(f"   ⚠️  Not found: {aspect_type_id} (already deleted)")
            not_found_count += 1
        else:
            print(f"   ❌ Error deleting {aspect_type_id}: {error_msg[:80]}")
            error_count += 1
    
    print()

print("=" * 60)
print("✅ Aspect Type deletion completed!")
print("=" * 60)
print(f"   Deleted: {deleted_count}")
print(f"   Not found: {not_found_count}")
print(f"   Errors: {error_count}")

## Section 4: Delete BigQuery Dataset

Finally, we'll delete the entire BigQuery dataset, which includes all ~278 census tables.

**⚠️  WARNING:** This will permanently delete all data in the `census_bureau_acs` dataset.

In [ ]:
# Delete BigQuery dataset
print("🗑️  Deleting BigQuery Dataset...")
print()

dataset_id = f"{PROJECT_ID}.{DATASET_ID}"

print(f"⚠️  About to delete dataset: {dataset_id}")
if total_tables > 0:
    print(f"   This will delete all {total_tables} tables and their data")
print()

try:
    # Delete the dataset with delete_contents=True to delete all tables
    bq_client.delete_dataset(
        dataset_id,
        delete_contents=True,  # Delete all tables in the dataset
        not_found_ok=True      # Don't error if already deleted
    )
    
    print("=" * 60)
    print("✅ BigQuery dataset deleted successfully!")
    print("=" * 60)
    print(f"   Dataset: {DATASET_ID}")
    if total_tables > 0:
        print(f"   All {total_tables} tables and data have been permanently deleted")
    
except Exception as e:
    error_msg = str(e)
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("⚠️  Dataset not found (already deleted)")
    else:
        print(f"❌ Error deleting dataset: {error_msg}")
        raise

## Section 5: Cleanup Summary

In [ ]:
# Summary
print()
print("=" * 70)
print("🎉 CLEANUP COMPLETED!")
print("=" * 70)
print()
print("📊 Summary of deleted resources:")
print()
print("   1. Dataplex Aspects:")
print(f"      - Removed from {total_tables} table entries")
print()
print("   2. Dataplex Aspect Types:")
print("      - census-survey-metadata")
print("      - data-governance-public")
print()
print("   3. BigQuery Dataset:")
print(f"      - {DATASET_ID} (with {total_tables} tables)")
print()
print("✅ All resources from config_and_data_setup.ipynb have been cleaned up!")
print()
print("🔗 Verify in Console:")
print(f"   BigQuery: https://console.cloud.google.com/bigquery?project={PROJECT_ID}")
print(f"   Dataplex Aspect Types: https://console.cloud.google.com/dataplex/govern/aspect-types?project={PROJECT_ID}")